# Bayesian Networks: Exact Inference and Utility-Based Decisions

Curated from completed Artificial Intelligence coursework at Sharif University of Technology.
Repository owner: Amirhossein Hashemi Shahroodi (`amirshs84`).
Course-provided prompts, data and scaffolding are not claimed as original work; see `../../ATTRIBUTION.md`.
Run cells from top to bottom in this notebook’s directory. See the root README for scope, dependencies and limitations.


# Welcome to Fmapp! & The Data Dump

Welcome to your first day as an AI  Engineer at **Fmapp!** *(Legal Disclaimer: Fmapp! is a completely fictional startup. Any resemblance to actual ride-hailing companies, living or dead, or any green-themed apps you might use to commute, is entirely coincidental. Please do not call our lawyers).*

Right now, the company has a massive problem: our surge pricing algorithm is deeply confused. It regularly triggers 2x price surges on completely empty streets (frustrating our passengers), and stays flat during rush hour gridlock (frustrating our drivers). The CEO has made it simple: fix the surge logic before we go bankrupt. We need a system that actually understands *why* traffic is happening and makes smart economic decisions. 

**Your Mission:** You are going to build a **Dynamic Surge & Economics Engine** from the ground up using a Bayesian Network. 

> ⚠️ **Rules of the Road:** We are calculating pure, mathematical truth here. You must implement everything using **exact inference** methods (like Variable Elimination or direct enumeration). **Do NOT** use MCMC, Gibbs, Rejection Sampling, Likelihood Weighting, or any other sampling approximations in this notebook.

## The Bayes Net Blueprint

Each variable is **Boolean** (`True` / `False`). Here's the generative story Fmapp!'s lead engineer swears is "basically causal":

| Node | Symbol | Meaning |
|------|--------|--------|
| `GameDay` | **G** | A local sports game is happening |
| `TrafficAccident` | **A** | Accident on the main highway |
| `HeavyTraffic` | **T** | Traffic is crawling |
| `HighDemand` | **D** | Passengers flooding the app |
| `LowSupply` | **S** | Drivers avoiding the zone |
| `SurgeRequired` | **Z** | Zone needs surge pricing |

**Dependencies (arrows point from parent → child):**

- **G** and **A** are roots (no parents).
- **T** depends on **G** and **A**.
- **D** depends on **G**.
- **S** depends on **T**.
- **Z** depends on **D** and **S**.

```
     GameDay (G)          TrafficAccident (A)
          |    \___________________/
          |                    \
          v                     v
    HighDemand (D)        HeavyTraffic (T)
          |                     |
          |                     v
          |               LowSupply (S)
          \___________   __________/
                      v v
               SurgeRequired (Z)
```

Over the next seven parts, you'll load real(ish) logs, sanity-check the graph, learn CPTs from data, run inference, play detective, and finally tell Product whether to flip the surge switch—or buy expensive traffic data.

---
# Part 1: Welcome to Fmapp! & The CSV Data Dump

It's 9:07 AM. Slack is on fire. The Data Engineering team just `@channel`'d you:

> *"Historical zone dump attached. 5,000 rows. Good luck lol."*

K wants probabilities **from data**, not vibes. But first—you need the data on your machine.

The cell below does two things:
1. **Synthesizes** a realistic 5,000-row historical log using Fmapp!'s *true* generative process (the CPTs the company believes—but never wrote down).
2. **Saves** it to `fmapp_historical_data.csv` so you're not tied to this notebook forever.

Run it once. Then load the CSV like a civilized engineer.

In [1]:
# =============================================================================
# FMAPP! DATA FACTORY — generates fmapp_historical_data.csv (5,000 rows)
# "We don't guess probabilities; we derive them from the database."
#           — Every data scientist, eventually contradicted by this cell.
# =============================================================================

import numpy as np
import pandas as pd
from pathlib import Path

RNG = np.random.default_rng(42)  # reproducibility: K loves reproducibility
N_ROWS = 5_000
CSV_PATH = Path("generated_surge_data.csv")

# --- True generative model (ground truth for this homework universe) ---
P_G = 0.15
P_A = 0.08

# P(T | G, A) — keys are (GameDay, TrafficAccident)
CPT_T = {
    (False, False): 0.12,
    (False, True):  0.65,
    (True,  False): 0.35,
    (True,  True):  0.80,
}

# P(D | G)
CPT_D = {False: 0.20, True: 0.55}

# P(S | T)
CPT_S = {False: 0.10, True: 0.45}

# P(Z | D, S)
CPT_Z = {
    (False, False): 0.05,
    (False, True):  0.40,
    (True,  False): 0.70,
    (True,  True):  0.92,
}

def _bernoulli(p, size):
    return RNG.random(size) < p

# Sample in topological order: G, A → T, D → S → Z
game_day = _bernoulli(P_G, N_ROWS)
traffic_accident = _bernoulli(P_A, N_ROWS)

p_t = np.array([CPT_T[(bool(g), bool(a))] for g, a in zip(game_day, traffic_accident)])
heavy_traffic = _bernoulli(p_t, N_ROWS)

p_d = np.array([CPT_D[bool(g)] for g in game_day])
high_demand = _bernoulli(p_d, N_ROWS)

p_s = np.array([CPT_S[bool(t)] for t in heavy_traffic])
low_supply = _bernoulli(p_s, N_ROWS)

p_z = np.array([CPT_Z[(bool(d), bool(s))] for d, s in zip(high_demand, low_supply)])
surge_required = _bernoulli(p_z, N_ROWS)

df_raw = pd.DataFrame({
    "GameDay": game_day,
    "TrafficAccident": traffic_accident,
    "HeavyTraffic": heavy_traffic,
    "HighDemand": high_demand,
    "LowSupply": low_supply,
    "SurgeRequired": surge_required,
})

df_raw.to_csv(CSV_PATH, index=False)
print(f"✅ Wrote {len(df_raw):,} rows to {CSV_PATH.resolve()}")
print(f"   Surge rate in dump: {df_raw['SurgeRequired'].mean():.3f} (CEO will ask)")

✅ Wrote 5,000 rows to /workspace/scratch/d1fea2b9d7f6/ai-algorithms-portfolio/notebooks/04_probabilistic_reasoning/generated_surge_data.csv
   Surge rate in dump: 0.265 (CEO will ask)


In [2]:
# --- Load the dump like a responsible adult ---
df = pd.read_csv(CSV_PATH)

# CSV stores 0/1; coerce to bool for sanity
BOOL_COLS = [
    "GameDay", "TrafficAccident", "HeavyTraffic",
    "HighDemand", "LowSupply", "SurgeRequired",
]
df[BOOL_COLS] = df[BOOL_COLS].astype(bool)

print("First five zone-states from the archive:")
df.head()

First five zone-states from the archive:
   GameDay  TrafficAccident  HeavyTraffic  HighDemand  LowSupply  SurgeRequired
0    False            False         False       False      False          False
1    False            False         False        True      False          False
2    False            False         False       False       True          False
3    False            False          True       False       True          False
4     True             True          True        True      False           True


---
# Part 2: The Blueprint (D-Separation) [20 points]

Before K approves your GPU budget, the **lead engineer** wants proof the graph isn't nonsense.

**Question:** Is `HighDemand` **conditionally independent** of `LowSupply` given `GameDay` and `HeavyTraffic`?

In symbols: does **D ⊥ S | G, T** hold in this network?

Answer using **d-separation rules** (markdown proof) **and** a **computational check** on `df` (e.g., compare conditional distributions). If the graph says they should be independent, empirical frequencies should be *close*—but not identical—because 5,000 rows is finite.

**Hint:** Trace all undirected paths between D and S. Watch for **forks**, **chains**, and **colliders**—and remember what happens when you **condition** on a node.

### Your d-separation analysis

We want to test whether:

$$D \perp S \mid G,T$$

where:

- $D$ = HighDemand
- $S$ = LowSupply
- $G$ = GameDay
- $T$ = HeavyTraffic

The relevant graph structure is:

$$G \rightarrow D$$
$$G \rightarrow T \leftarrow A$$
$$T \rightarrow S$$
$$D \rightarrow Z \leftarrow S$$

Now examine all undirected paths between $D$ and $S$.

### Path 1
$$D \leftarrow G \rightarrow T \rightarrow S$$

- $G$ is a non-collider and is observed.
- $T$ is also a non-collider and is observed.

Conditioning on a non-collider blocks the path.

Therefore this path is **blocked**.

---

### Path 2
$$D \rightarrow Z \leftarrow S$$

- $Z$ is a collider.
- Neither $Z$ nor any descendant of $Z$ is observed.

A collider blocks the path unless conditioned on.

Therefore this path is also **blocked**.

---

Since every path between $D$ and $S$ is blocked after conditioning on $G$ and $T$, the variables are d-separated.

## Conclusion

$$D \perp S \mid G,T$$

So the network predicts that `HighDemand` and `LowSupply` should be conditionally independent given `GameDay` and `HeavyTraffic`.


In [3]:
# ### TODO ### Part 2 — Computational sanity check
#
# Verify whether P(S | G, T, D) ≈ P(S | G, T) using the dataframe.
# If D and S are conditionally independent given G and T, these should match closely.

results = []

for g in [False, True]:
    for t in [False, True]:
        subset_gt = df[(df["GameDay"] == g) & (df["HeavyTraffic"] == t)]

        p_s_given_gt = subset_gt["LowSupply"].mean()

        for d in [False, True]:
            subset_gtd = subset_gt[subset_gt["HighDemand"] == d]

            if len(subset_gtd) == 0:
                p_s_given_gtd = float("nan")
            else:
                p_s_given_gtd = subset_gtd["LowSupply"].mean()

            results.append({
                "GameDay": g,
                "HeavyTraffic": t,
                "HighDemand": d,
                "P(S|G,T)": round(p_s_given_gt, 4),
                "P(S|G,T,D)": round(p_s_given_gtd, 4),
                "Difference": round(abs(p_s_given_gt - p_s_given_gtd), 4),
            })

results_df = pd.DataFrame(results)
print(results_df)

print("\nInterpretation:")
print("Small differences support the d-separation claim that D ⟂ S | G,T.")


   GameDay  HeavyTraffic  HighDemand  P(S|G,T)  P(S|G,T,D)  Difference
0    False         False       False    0.1038      0.1077      0.0039
1    False         False        True    0.1038      0.0881      0.0156
2    False          True       False    0.4604      0.4754      0.0150
3    False          True        True    0.4604      0.3985      0.0619
4     True         False       False    0.0996      0.1005      0.0009
5     True         False        True    0.0996      0.0988      0.0008
6     True          True       False    0.4267      0.4222      0.0044
7     True          True        True    0.4267      0.4303      0.0036

Interpretation:
Small differences support the d-separation claim that D ⟂ S | G,T.


---
# Part 3: The CPT Builder (Data → Probabilities) [20 points]

Raw logs are not a Bayes Net. **Conditional Probability Tables (CPTs)** are.

Fmapp!'s rule: *estimate each CPT by counting* (maximum likelihood from the dump). No hand-wavy priors—unless you're the CEO, in which case hand-wavy is a feature.

Your first construction task: learn **P(HeavyTraffic | GameDay, TrafficAccident)** from `df`.

In [4]:
# --- Boilerplate: helper to pretty-print a CPT (not your TODO) ---

def print_cpt_table(cpt_dict, child_name="HeavyTraffic"):
    """cpt_dict maps parent assignment tuples -> P(child=True)."""
    print(f"\nCPT for {child_name} (probability of True):")
    print(f"{'Parent assignment':<40} P({child_name}=True)")
    print("-" * 55)
    for parents, p in sorted(cpt_dict.items()):
        print(f"{str(parents):<40} {p:.4f}")


# ### TODO ### Part 3 — Build CPT for HeavyTraffic | GameDay, TrafficAccident

def estimate_cpt_heavy_traffic(dataframe: pd.DataFrame) -> dict:
    """
    Returns a dict:
        (game_day: bool, traffic_accident: bool) -> P(HeavyTraffic=True | ...)
    Estimate from counts in `dataframe`. Handle empty bins gracefully (e.g., return 0.0).
    """

    cpt = {}

    for g in [False, True]:
        for a in [False, True]:

            subset = dataframe[
                (dataframe["GameDay"] == g) &
                (dataframe["TrafficAccident"] == a)
            ]

            if len(subset) == 0:
                prob = 0.0
            else:
                prob = subset["HeavyTraffic"].mean()

            cpt[(g, a)] = prob

    return cpt


cpt_heavy_traffic = estimate_cpt_heavy_traffic(df)
print_cpt_table(cpt_heavy_traffic)

# Ground-truth targets from the generative cell (for your eyes only after you implement):
# (F,F)->0.12  (F,T)->0.65  (T,F)->0.35  (T,T)->0.80



CPT for HeavyTraffic (probability of True):
Parent assignment                        P(HeavyTraffic=True)
-------------------------------------------------------
(False, False)                           0.1180
(False, True)                            0.6361
(True, False)                            0.3599
(True, True)                             0.8254


---
# Part 4: The Baseline (Exact Inference by Enumeration) [30 points]

It's a standard Tuesday. No disasters on Slack (yet). You need the **baseline** probability that surge is required.

**Query:**  
$$P(\text{SurgeRequired} = \text{True} \mid \text{HighDemand} = \text{True},\ \text{TrafficAccident} = \text{False})$$

Compute an exact posterior by enumerating the Boolean network. This submitted implementation uses enumeration, not factor-based variable elimination. Use the **true CPTs** below (same as the data factory)—in practice you'd plug in your Part 3 estimates, but exact inference needs a fully specified network.

> 💡 **Factor tip:** Represent each CPT as a dictionary mapping **assignments** (tuples of `(var, value)` pairs) to probabilities. Multiplying factors = join on shared variables; summing out = marginalize.

In [5]:
# =============================================================================
# Network specification for exact inference (boilerplate — do not skip reading)
# =============================================================================

VARIABLES = [
    "GameDay", "TrafficAccident", "HeavyTraffic",
    "HighDemand", "LowSupply", "SurgeRequired",
]

# Short names for tired engineers
G, A, T, D, S, Z = VARIABLES

NETWORK_CPTS = {
    G: {"parents": [], "table": {(): 0.15}},  # P(G=True)
    A: {"parents": [], "table": {(): 0.08}},
    T: {
        "parents": [G, A],
        "table": {
            (False, False): 0.12,
            (False, True):  0.65,
            (True,  False): 0.35,
            (True,  True):  0.80,
        },
    },
    D: {
        "parents": [G],
        "table": {False: 0.20, True: 0.55},
    },
    S: {
        "parents": [T],
        "table": {False: 0.10, True: 0.45},
    },
    Z: {
        "parents": [D, S],
        "table": {
            (False, False): 0.05,
            (False, True):  0.40,
            (True,  False): 0.70,
            (True,  True):  0.92,
        },
    },
}


def make_factor(var, cpt_entry, evidence=None):
    """
    Build a factor for `var` from NETWORK_CPTS[var].
    Returns dict: frozenset of (var, bool) assignments -> float
    Only includes entries where `var` is True OR False (full distribution).
    """
    evidence = evidence or {}
    parents = cpt_entry["parents"]
    table = cpt_entry["table"]
    factor = {}

    if not parents:
        p_true = table[()]
        for val in [False, True]:
            if var in evidence and evidence[var] != val:
                continue
            assign = frozenset([(var, val)])
            factor[assign] = p_true if val else (1 - p_true)
        return factor

    # Enumerate parent combinations (respecting evidence on parents)
    parent_vals_list = []
    for p in parents:
        if p in evidence:
            parent_vals_list.append([evidence[p]])
        else:
            parent_vals_list.append([False, True])

    from itertools import product
    for parent_combo in product(*parent_vals_list):
        key = parent_combo[0] if len(parent_combo) == 1 else tuple(parent_combo)
        p_true = table[key]
        parent_dict = dict(zip(parents, parent_combo))
        for val in [False, True]:
            if var in evidence and evidence[var] != val:
                continue
            assign_dict = {**parent_dict, var: val}
            assign = frozenset(assign_dict.items())
            factor[assign] = p_true if val else (1 - p_true)
    return factor


def factor_scope(factor):
    scope = set()
    for assign in factor:
        scope.update(v for v, _ in assign)
    return scope


def multiply_factors(f1, f2):
    """Pointwise multiply two factors (join on shared variables)."""
    shared = factor_scope(f1) & factor_scope(f2)
    result = {}
    for a1, p1 in f1.items():
        d1 = dict(a1)
        for a2, p2 in f2.items():
            d2 = dict(a2)
            if all(d1.get(v) == d2.get(v) for v in shared):
                merged = frozenset({**d1, **d2}.items())
                result[merged] = p1 * p2
    return result


def restrict_factor(factor, evidence):
    """Hard-condition factor on evidence."""
    out = {}
    for assign, p in factor.items():
        d = dict(assign)
        if all(d.get(v) == ev for v, ev in evidence.items()):
            out[assign] = p
    return out


def sum_out(factor, var):
    """Marginalize `var` from factor."""
    result = {}
    for assign, p in factor.items():
        d = dict(assign)
        d.pop(var, None)
        new_assign = frozenset(d.items())
        result[new_assign] = result.get(new_assign, 0) + p
    return result


def normalize(dist):
    total = sum(dist.values())
    return {k: v / total for k, v in dist.items()} if total else dist

In [6]:
# ### TODO ### Part 4 — Variable Elimination
#
# Implement exact_posterior(query_var, evidence, elimination_order=None)
# that returns P(query_var=True | evidence).
#
# Suggested steps:
#   1. Build one factor per CPT (use make_factor + evidence)
#   2. Multiply all factors
#   3. For each var in elimination_order (that is not query or evidence): sum out
#   4. Normalize remaining distribution over query_var

from itertools import product

def joint_probability(full_assignment):
    """Compute full joint probability for a complete assignment."""

    prob = 1.0

    for var in VARIABLES:

        entry = NETWORK_CPTS[var]
        parents = entry["parents"]
        table = entry["table"]

        if not parents:
            p_true = table[()]
        else:
            key = tuple(full_assignment[p] for p in parents)
            if len(parents) == 1:
                key = key[0]
            p_true = table[key]

        prob *= p_true if full_assignment[var] else (1 - p_true)

    return prob


def exact_posterior(query_var, evidence, elimination_order=None):
    """
    evidence: dict like {D: True, A: False}
    elimination_order: list of variables to eliminate (hidden vars only)
    Returns: float, P(query_var=True | evidence)

    NOTE:
    For this small Boolean network we implement exact inference by
    enumerating hidden variables directly. This produces the same
    exact posterior as variable elimination.
    """

    hidden_vars = [
        v for v in VARIABLES
        if v not in evidence and v != query_var
    ]

    numerator = 0.0
    denominator = 0.0

    for values in product([False, True], repeat=len(hidden_vars) + 1):

        query_value = values[0]
        hidden_values = values[1:]

        assignment = {
            query_var: query_value,
            **evidence,
            **dict(zip(hidden_vars, hidden_values))
        }

        p = joint_probability(assignment)

        denominator += p

        if query_value:
            numerator += p

    return numerator / denominator


EVIDENCE_PART4 = {D: True, A: False}
# Good elimination orders eliminate hidden vars before query/evidence vars
ELIM_ORDER = [G, T, S]  # you may adjust if your implementation needs it

p_surge_given_demand_no_accident = exact_posterior(Z, EVIDENCE_PART4, ELIM_ORDER)
print(f"P(SurgeRequired=True | HighDemand=True, TrafficAccident=False) = {p_surge_given_demand_no_accident:.4f}")


P(SurgeRequired=True | HighDemand=True, TrafficAccident=False) = 0.7370


---
# Part 5: The Detective (MAP Inference) [30 points]

Support ticket #8841:

> *"Surge is 1.8x and there are ZERO drivers. WHAT HAPPENED??"*

You observe:
- `SurgeRequired = True`
- `LowSupply = True`

K wants the **most likely root-cause explanation** among the hidden variables **`GameDay`** and **`TrafficAccident`**.

**MAP query:**  
$$\arg\max_{g,a} P(g, a \mid Z{=}\text{True}, S{=}\text{True})$$

Use **exact** inference (enumeration over the 4 assignments is totally fine here—this is a tiny network; K doesn't need to see you brag about scalability yet).

In [7]:
# ### TODO ### Part 5 — MAP over root causes
#
# Return the assignment (GameDay, TrafficAccident) with highest posterior
# given evidence Z=True, S=True.

from itertools import product

def map_root_causes(evidence_zs):
    """
    evidence_zs: e.g. {Z: True, S: True}
    Returns: dict with keys 'GameDay', 'TrafficAccident' (bool values),
             and 'posterior' (float) for the winning assignment.
    """

    scores = {}

    for g, a in product([False, True], repeat=2):

        assignment = {G: g, A: a, **evidence_zs}

        hidden_vars = [T, D]

        prob = 0.0

        for t, d in product([False, True], repeat=2):

            full = {
                G: g,
                A: a,
                T: t,
                D: d,
                S: evidence_zs[S],
                Z: evidence_zs[Z],
            }

            p = 1.0

            p *= NETWORK_CPTS[G]["table"][()] if full[G] else (1 - NETWORK_CPTS[G]["table"][()])
            p *= NETWORK_CPTS[A]["table"][()] if full[A] else (1 - NETWORK_CPTS[A]["table"][()])

            p_t = NETWORK_CPTS[T]["table"][(full[G], full[A])]
            p *= p_t if full[T] else (1 - p_t)

            p_d = NETWORK_CPTS[D]["table"][full[G]]
            p *= p_d if full[D] else (1 - p_d)

            p_s = NETWORK_CPTS[S]["table"][full[T]]
            p *= p_s if full[S] else (1 - p_s)

            p_z = NETWORK_CPTS[Z]["table"][(full[D], full[S])]
            p *= p_z if full[Z] else (1 - p_z)

            prob += p

        scores[(g, a)] = prob

    normalized = normalize(scores)

    best_assignment = max(normalized, key=normalized.get)

    return {
        "GameDay": best_assignment[0],
        "TrafficAccident": best_assignment[1],
        "posterior": normalized[best_assignment],
    }


map_result = map_root_causes({Z: True, S: True})
print("Most likely root-cause assignment:")
print(f"  GameDay          = {map_result['GameDay']}")
print(f"  TrafficAccident  = {map_result['TrafficAccident']}")
print(f"  Posterior mass   = {map_result['posterior']:.6f}")


Most likely root-cause assignment:
  GameDay          = False
  TrafficAccident  = False
  Posterior mass   = 0.612442


---
# Part 6: The Economics (Value of Perfect Information) [+10 points additional]

Vendor pitch deck: **City Traffic API** — $5,000/month, perfectly reveals `TrafficAccident` in real time.

K's question: *"Does knowing accidents early actually change our surge decisions enough to pay for this?"*

That's **Value of Perfect Information (VPI)** for node **A**.

**Decision:** predict whether to treat the zone as needing surge (`Z`), before seeing the day's full picture.  
**Utility** (Fmapp! finance team's napkin):

| Your action \ True state | Z=False (no surge needed) | Z=True (surge needed) |
|--------------------------|---------------------------|------------------------|
| **Surge** (activate)     | -$800 (angry passengers)  | +$2,500 (balanced market) |
| **Normal** (no surge)    | +$200 (smooth ops)        | -$3,000 (drivers flee) |

You may observe **`HighDemand = True`** before deciding (Tuesday baseline scenario). Without the API, you don't know **A**. With perfect info on **A**, you know it.

**VPI(A)** = Expected utility with perfect knowledge of A − Expected utility with no extra info about A (optimal decision in each case).

In [8]:
# --- Utility boilerplate (Part 6) ---

UTILITY = {
    # (action, true_Z): payoff in USD
    ("surge",  False): -800,
    ("surge",  True):  2500,
    ("normal", False):  200,
    ("normal", True):  -3000,
}
ACTIONS = ["surge", "normal"]
MONTHLY_API_COST = 5000  # K will compare your VPI to this


def expected_utility_of_action(p_z_true, action):
    """EU(action) = sum_z P(z) * U(action, z)"""
    return (p_z_true * UTILITY[(action, True)]
            + (1 - p_z_true) * UTILITY[(action, False)])


def optimal_action_and_eu(p_z_true):
    """Return (best_action, best_eu)."""
    best = max(ACTIONS, key=lambda a: expected_utility_of_action(p_z_true, a))
    return best, expected_utility_of_action(p_z_true, best)


# ### TODO ### Part 6 — Value of Perfect Information for TrafficAccident
#
# Given evidence E = {HighDemand: True}:
#   1. Compute EU* without observing A (marginalize A out of P(Z|E))
#   2. For each a in {False, True}, compute P(Z | E, A=a), pick optimal action, weight by P(A=a|E)
#   3. VPI = EU_with_perfect_A - EU_without_A
#   4. Print whether the API is worth $5,000/month (compare VPI to cost; state assumptions)

def vpi_traffic_accident(evidence):
    """
    Returns dict with keys:
      'eu_without_info', 'eu_with_perfect_info', 'vpi',
      'worth_api' (bool), 'recommendation' (str)
    Use variable_elimination (or exact enumeration) for all probabilities.
    """

    hidden_order = [G, T, S]

    # Without observing A
    p_z = exact_posterior(Z, evidence, hidden_order + [A])
    _, eu_without = optimal_action_and_eu(p_z)

    # P(A | evidence)
    p_a_true = exact_posterior(A, evidence, [G, T, S, Z])
    p_a = {True: p_a_true, False: 1 - p_a_true}

    eu_with_info = 0.0

    for a_val in [False, True]:

        extended_evidence = {**evidence, A: a_val}

        p_z_given_a = exact_posterior(Z, extended_evidence, hidden_order)

        _, best_eu = optimal_action_and_eu(p_z_given_a)

        eu_with_info += p_a[a_val] * best_eu

    vpi = eu_with_info - eu_without

    worth_api = vpi > MONTHLY_API_COST

    recommendation = (
        "API is financially worth it."
        if worth_api
        else "API is NOT worth the monthly cost."
    )

    return {
        "eu_without_info": eu_without,
        "eu_with_perfect_info": eu_with_info,
        "vpi": vpi,
        "worth_api": worth_api,
        "recommendation": recommendation,
    }


vpi_report = vpi_traffic_accident({D: True})
print(f"EU without knowing A:     ${vpi_report['eu_without_info']:,.2f}")
print(f"EU with perfect A info:   ${vpi_report['eu_with_perfect_info']:,.2f}")
print(f"VPI(TrafficAccident):     ${vpi_report['vpi']:,.2f}")
print(f"API costs per month:      ${MONTHLY_API_COST:,}")
print(vpi_report["recommendation"])


EU without knowing A:     $1,642.43
EU with perfect A info:   $1,642.43
VPI(TrafficAccident):     $0.00
API costs per month:      $5,000
API is NOT worth the monthly cost.


---
# Part 7: The Product Decision (Expected Utility Engine) [+10 points additional]

Algorithms are worthless if Product can't ship them. Build **`trigger_surge_action(evidence_dict)`**:

1. Run **exact inference** for $P(Z \mid \text{evidence})$.
2. Compute **Expected Utility** of surging vs. staying normal (use the matrix below).
3. If EU(surge) > EU(normal) → print the surge banner. Else → print the chill banner.

This is what runs in the zone controller (minus the emojis, but we kept them for morale).

In [9]:
# --- Product utility matrix (Part 7) — slightly different stakes, same structure ---

PRODUCT_UTILITY = {
    ("surge",  False): -600,
    ("surge",  True):  2200,
    ("normal", False):  150,
    ("normal", True):  -2800,
}


def product_eu(p_z_true, action):
    return (p_z_true * PRODUCT_UTILITY[(action, True)]
            + (1 - p_z_true) * PRODUCT_UTILITY[(action, False)])


# ### TODO ### Part 7 — Expected Utility surge engine

def trigger_surge_action(evidence_dict):
    """
    evidence_dict: maps variable names (strings) to bool, e.g.
        {"HighDemand": True, "TrafficAccident": False}
    Uses exact inference + PRODUCT_UTILITY to decide.
    """

    hidden_vars = [v for v in VARIABLES if v not in evidence_dict and v != Z]

    p_z_true = exact_posterior(Z, evidence_dict, hidden_vars)

    eu_surge = product_eu(p_z_true, "surge")
    eu_normal = product_eu(p_z_true, "normal")

    if eu_surge > eu_normal:
        decision = "surge"
        banner = "🚀 Surge pricing ENABLED"
    else:
        decision = "normal"
        banner = "😌 Keep normal pricing"

    print("=" * 60)
    print("Evidence:", evidence_dict)
    print(f"P(SurgeRequired=True | evidence) = {p_z_true:.4f}")
    print(f"EU(surge)  = {eu_surge:.2f}")
    print(f"EU(normal) = {eu_normal:.2f}")
    print(banner)

    return {
        "decision": decision,
        "p_z_true": p_z_true,
        "eu_surge": eu_surge,
        "eu_normal": eu_normal,
    }


# --- Demo calls (uncomment after you implement) ---
trigger_surge_action({"HighDemand": True, "TrafficAccident": False})
trigger_surge_action({"HighDemand": False, "LowSupply": False})


Evidence: {'HighDemand': True, 'TrafficAccident': False}
P(SurgeRequired=True | evidence) = 0.7370
EU(surge)  = 1463.67
EU(normal) = -2024.23
🚀 Surge pricing ENABLED
Evidence: {'HighDemand': False, 'LowSupply': False}
P(SurgeRequired=True | evidence) = 0.0500
EU(surge)  = -460.00
EU(normal) = 2.50
😌 Keep normal pricing
{'decision': 'normal', 'p_z_true': 0.05000000000000001, 'eu_surge': -460.0, 'eu_normal': 2.4999999999999716}
